In [1]:
# Libraries.
import pandas as pd
import numpy as np

In [2]:
# Read in data.
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")
orthologTable = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/orthologTable.txt", sep="\t")
orthologTableIDs = orthologTable[~orthologTable["Mouse gene stable ID"].isna()].loc[:, ["Gene stable ID", "Mouse gene stable ID"]]

In [3]:
gtexEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [4]:
emtabEP.sum()

Brain                                                1000000.0
Colon                                                1000000.0
Esophagus                                            1000000.0
Heart                                                1000000.0
Kidney                                               1000000.0
Liver                                                1000000.0
Pancreas                                             1000000.0
Stomach                                              1000000.0
Gene type    protein_codingprotein_codingprotein_codinglncR...
dtype: object

In [5]:
orthologTable

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score
0,ENSG00000198888,ENSMUSG00000064341,ortholog_one2one,50.0
1,ENSG00000198763,ENSMUSG00000064345,ortholog_one2one,75.0
2,ENSG00000198804,ENSMUSG00000064351,ortholog_one2one,100.0
3,ENSG00000198712,ENSMUSG00000064354,ortholog_one2one,100.0
4,ENSG00000228253,ENSMUSG00000064356,ortholog_one2one,100.0
...,...,...,...,...
28009,ENSG00000081692,ENSMUSG00000036819,ortholog_one2one,75.0
28010,ENSG00000157873,ENSMUSG00000022074,ortholog_one2many,0.0
28011,ENSG00000157873,ENSMUSG00000042333,ortholog_one2many,100.0
28012,ENSG00000132676,ENSMUSG00000068921,ortholog_one2one,75.0


In [6]:
orthologTest = tuple(orthologTableIDs.iloc[0, :])
humanOrtholog = gtexEP[gtexEP.index.isin(orthologTest)]
mouseOrtholog = emtabEP[emtabEP.index.isin(orthologTest)]

In [7]:
# Euclidean distance. My own methodology.
def euclideanDist(humanOrtholog, mouseOrtholog):
    distSum = 0
    for i in range(0, humanOrtholog.shape[1] - 1):
        distSum += np.square(humanOrtholog.iloc[0, i] - mouseOrtholog.iloc[0, i])
    return np.sqrt(distSum)

In [8]:
euclideanDist(humanOrtholog, mouseOrtholog)

np.float64(44492.547301776765)

In [9]:
# Euclidean Distance. Method using numpy's euclidean distance formula.
np.linalg.norm(np.array(humanOrtholog.iloc[:, :-1]) - np.array(mouseOrtholog.iloc[:, :-1]))

np.float64(44492.547301776765)

In [10]:
# Pearson Distance. My own methodology.
def pearsonDist(humanOrtholog, mouseOrtholog):
    # Calculates the Z-Scores for our vectors, and uses these Z-Score vectors to calculate Pearson Distance. The formula was provided in Piasecka et al. 2012.
    ZxT = ((humanOrtholog.iloc[:, :-1] - np.mean(humanOrtholog.iloc[:, :-1])) / np.std(humanOrtholog.iloc[:, :-1])).T
    Zy = (mouseOrtholog.iloc[:, :-1] - np.mean(mouseOrtholog.iloc[:, :-1])) / np.std(mouseOrtholog.iloc[:, :-1])
    return 1 - ((Zy.dot(ZxT) / ZxT.shape[0]).iloc[0, 0])

In [11]:
pearsonDist(humanOrtholog, mouseOrtholog)

np.float64(0.2094305670992722)

In [12]:
# Pearson Distance is 1 - r, and I found online that r is just the correlation matrix between two dataframes. So I tried that method.
corr = humanOrtholog.iloc[:, :-1].reset_index(drop=True).corrwith(mouseOrtholog.iloc[:, :-1].reset_index(drop=True), method="pearson", axis=1)
1 - corr[0]

np.float64(0.20943055191829985)

In [13]:
# TEC
def TEC(humanOrtholog, mouseOrtholog):
    # Turns the vectors binary. So if the expression is greater than 1, we consider that "expressed."
    humanOrthoBinary = (humanOrtholog.iloc[:, :-1] > 1).iloc[0, :]
    mouseOrthoBinary = (mouseOrtholog.iloc[:, :-1] > 1).iloc[0, :]

    humanOnlyTissueNum = ((humanOrthoBinary ^ mouseOrthoBinary) & humanOrthoBinary).sum()
    mouseOnlyTissueNum = ((mouseOrthoBinary ^ humanOrthoBinary) & mouseOrthoBinary).sum()

    return ((humanOnlyTissueNum / 8) + (mouseOnlyTissueNum / 8)) / 2


In [14]:
# This cell just gets the distance and TEC values, so I can append them to the expression profiles later.
myEuclideanDistArr = []
myPearsonDistArr = []
myTECArr = []
for i in range(0, orthologTableIDs.shape[0]):
    orthologTest = tuple(orthologTableIDs.iloc[i, :])
    humanOrtholog = gtexEP[gtexEP.index.isin(orthologTest)]
    mouseOrtholog = emtabEP[emtabEP.index.isin(orthologTest)]

    if not humanOrtholog.empty and not mouseOrtholog.empty:
        myEuclideanDistArr.append((orthologTest[0], orthologTest[1], euclideanDist(humanOrtholog, mouseOrtholog)))
        myPearsonDistArr.append((orthologTest[0], orthologTest[1], pearsonDist(humanOrtholog, mouseOrtholog)))
        myTECArr.append((orthologTest[0], orthologTest[1], TEC(humanOrtholog, mouseOrtholog)))

In [15]:
myEuclidDistDF = pd.DataFrame(myEuclideanDistArr, columns=["Human ID", "Mouse ID", "EuclidDist"])
myPearDistDF = pd.DataFrame(myPearsonDistArr, columns=["Human ID", "Mouse ID", "PearDist"])
myTECDF = pd.DataFrame(myTECArr, columns=["Human ID", "Mouse ID", "TEC"])

In [16]:
# The next 4 cells merge the expression profile dataframe with each distance and TEC column.
orthologTableEuclid = orthologTable.merge(myEuclidDistDF.loc[:, ["Human ID", "EuclidDist"]].groupby("Human ID").mean().reset_index(), left_on="Gene stable ID", right_on="Human ID", how="outer").set_index("Human ID")
orthologTableEuclidPear = orthologTableEuclid.merge(myPearDistDF.loc[:, ["Human ID", "PearDist"]].groupby("Human ID").mean().reset_index(), left_on="Gene stable ID", right_on="Human ID", how="outer").set_index("Human ID")
orthologTableEuclidPearTEC = orthologTableEuclidPear.merge(myTECDF.loc[:, ["Human ID", "TEC"]].groupby("Human ID").mean().reset_index(), left_on="Gene stable ID", right_on="Human ID", how="outer").set_index("Human ID")
orthologTableEuclidPearTEC.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.csv", index=True)
orthologTableEuclidPearTEC.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet", index=True)

In [17]:
orthologTableEuclidPearTEC

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,PearDist,TEC
Human ID,,,,,,,
ENSG00000000003,ENSG00000000003,ENSMUSG00000067377,ortholog_one2one,100.0,44.054842,1.130272,0.0000
ENSG00000000005,ENSG00000000005,ENSMUSG00000031250,ortholog_one2one,100.0,5.172072,1.141118,0.1250
ENSG00000000419,ENSG00000000419,ENSMUSG00000078919,ortholog_one2one,100.0,188.287732,1.211158,0.0000
ENSG00000000457,ENSG00000000457,ENSMUSG00000026584,ortholog_one2one,100.0,37.114733,0.814870,0.0000
ENSG00000000460,ENSG00000000460,ENSMUSG00000041406,ortholog_one2one,0.0,9.113600,0.959842,0.1875
...,...,...,...,...,...,...,...
NaN,ENSG00000310576,ENSMUSG00000035595,ortholog_one2one,100.0,NaN,NaN,NaN
NaN,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN
NaN,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
gtexEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [27]:
gtexEPNormalized = gtexEP.iloc[:, :-1].apply(lambda x: x / np.sqrt(x.sum()))
gtexEPNormalized["Gene type"] = gtexEP["Gene type"]
gtexEPNormalized

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,0.005800,0.022906,0.018108,0.003414,0.015746,0.023314,0.007128,0.010594,protein_coding
ENSG00000000005,0.000169,0.000754,0.000284,0.000258,0.000956,0.000019,0.000050,0.000207,protein_coding
ENSG00000000419,0.021388,0.040515,0.043499,0.024609,0.025236,0.022603,0.022147,0.037476,protein_coding
ENSG00000000457,0.002830,0.006585,0.006015,0.001902,0.003748,0.004093,0.003045,0.005296,protein_coding
ENSG00000000460,0.001348,0.002232,0.002199,0.000690,0.000989,0.001240,0.000699,0.001596,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [28]:
emtabEPNormalized = emtabEP.iloc[:, :-1].apply(lambda x: x / np.sqrt(x.sum()))
emtabEPNormalized["Gene type"] = emtabEP["Gene type"]
emtabEPNormalized

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSMUSG00000000001,0.034911,0.129181,0.143271,0.072898,0.072920,0.045745,0.008902,0.054003,protein_coding
ENSMUSG00000000003,0.000000,0.000000,0.000000,0.000054,0.000000,0.000000,0.000000,0.000000,protein_coding
ENSMUSG00000000028,0.002236,0.011324,0.005613,0.022680,0.002280,0.000872,0.000182,0.003491,protein_coding
ENSMUSG00000000031,0.002567,0.001138,2.650257,0.017373,0.000759,0.001307,0.001672,0.002287,lncRNA
ENSMUSG00000000037,0.001227,0.002676,0.003285,0.000775,0.000334,0.000011,0.000000,0.000288,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSMUSG00000109574,0.000333,0.000047,0.000022,0.000015,0.000008,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109575,0.001099,0.000000,0.000000,0.000009,0.000000,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109576,0.000023,0.000000,0.000000,0.000024,0.000013,0.000000,0.000000,0.000000,TEC


In [42]:
myEuclideanDistNormArr = []
myPearsonDistNormArr = []
for i in range(0, orthologTableIDs.shape[0]):
    orthologTest = tuple(orthologTableIDs.iloc[i, :])
    humanOrtholog = gtexEPNormalized[gtexEPNormalized.index.isin(orthologTest)]
    mouseOrtholog = emtabEPNormalized[emtabEPNormalized.index.isin(orthologTest)]

    if not humanOrtholog.empty and not mouseOrtholog.empty:
        myEuclideanDistNormArr.append((orthologTest[0], orthologTest[1], euclideanDist(humanOrtholog, mouseOrtholog)))
        myPearsonDistNormArr.append((orthologTest[0], orthologTest[1], pearsonDist(humanOrtholog, mouseOrtholog)))

myEuclidDistNormDF = pd.DataFrame(myEuclideanDistNormArr, columns=["Human ID", "Mouse ID", "EuclidDistNorm"])
myPearDistNormDF = pd.DataFrame(myPearsonDistNormArr, columns=["Human ID", "Mouse ID", "PearDistNorm"])


orthologTableEuclidPearTECNorm = orthologTableEuclidPearTEC.merge(myEuclidDistNormDF.loc[:, ["Human ID", "EuclidDistNorm"]].groupby("Human ID").mean().reset_index(), left_on="Gene stable ID", right_on="Human ID", how="outer").set_index("Human ID")
orthologTableEuclidPearTECNorm = orthologTableEuclidPearTECNorm.merge(myPearDistNormDF.loc[:, ["Human ID", "PearDistNorm"]].groupby("Human ID").mean().reset_index(), left_on="Gene stable ID", right_on="Human ID", how="outer").set_index("Human ID")

In [ ]:
newColOrder = list(orthologTableEuclidPearTECNorm.columns[:5]) + list(orthologTableEuclidPearTECNorm.columns[-2:-1]) + list(orthologTableEuclidPearTECNorm.columns[5:6]) + list(orthologTableEuclidPearTECNorm.columns[-1:]) + list(orthologTableEuclidPearTECNorm.columns[6:7])
orthologTableEuclidPearTECNorm = orthologTableEuclidPearTECNorm.loc[:, newColOrder]

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,EuclidDistNorm,PearDist,PearDistNorm,TEC
Human ID,,,,,,,,,
ENSG00000000003,ENSG00000000003,ENSMUSG00000067377,ortholog_one2one,100.0,44.054842,0.044055,1.130272,1.130272,0.0000
ENSG00000000005,ENSG00000000005,ENSMUSG00000031250,ortholog_one2one,100.0,5.172072,0.005172,1.141118,1.141118,0.1250
ENSG00000000419,ENSG00000000419,ENSMUSG00000078919,ortholog_one2one,100.0,188.287732,0.188288,1.211158,1.211158,0.0000
ENSG00000000457,ENSG00000000457,ENSMUSG00000026584,ortholog_one2one,100.0,37.114733,0.037115,0.814870,0.814870,0.0000
ENSG00000000460,ENSG00000000460,ENSMUSG00000041406,ortholog_one2one,0.0,9.113600,0.009114,0.959842,0.959842,0.1875
...,...,...,...,...,...,...,...,...,...
NaN,ENSG00000310576,ENSMUSG00000035595,ortholog_one2one,100.0,NaN,NaN,NaN,NaN,NaN
NaN,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NaN,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
orthologTableEuclidPearTECNorm.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.csv")
orthologTableEuclidPearTECNorm.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet")